<div style="background-color:#1A0D2E; border-left:7px solid #B537F2; padding:25px; border-radius:14px;">
<h1 style="color:#FFFFFF; font-size:36px; text-shadow:0 0 6px #B537F2; margin:0;">
🧪 Hands-On Lab: Building a Three-Way Split
</h1>
</div>

<div style="background-color:#1A0D2E; border-left:5px solid #B537F2; padding:15px 20px; border-radius:10px; margin-top:10px;">
<h2 style="color:#FFFFFF; font-size:24px; text-shadow:0 0 6px #bb7adc; margin:0;">
Step 1: 60/20/20 Train / Validation / Test Split
</h2>
</div>

In [4]:
from sklearn.model_selection import train_test_split
import pandas as pd
df = pd.read_csv("titanic.csv")
features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
df_clean = df[features + ["Survived"]].dropna()

X = df_clean[features]
y = df_clean["Survived"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (428, 5)
X_val shape: (143, 5)
X_test shape: (143, 5)


<div style="background-color:#1A0D2E; border-left:5px solid #B537F2; padding:15px 20px; border-radius:10px; margin-top:10px;">
<h2 style="color:#FFFFFF; font-size:24px; text-shadow:0 0 6px #bb7adc; margin:0;">
Step 2: Train a Model and Tune Using the Validation Set
</h2>
</div>

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

for depth in [2, 4, 6, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    val_preds = model.predict(X_val)
    val_acc = accuracy_score(y_val, val_preds)
    print(f"max_depth={depth} -> validation accuracy: {val_acc:.4f}")

max_depth=2 -> validation accuracy: 0.6783
max_depth=4 -> validation accuracy: 0.6923
max_depth=6 -> validation accuracy: 0.7273
max_depth=None -> validation accuracy: 0.6923


<div style="background-color:#1A0D2E; border-left:5px solid #B537F2; padding:15px 20px; border-radius:10px; margin-top:10px;">
<h2 style="color:#FFFFFF; font-size:24px; text-shadow:0 0 6px #bb7adc; margin:0;">
Step 3: Final Evaluation on the Test Set
</h2>
</div>

In [6]:
final_model = DecisionTreeClassifier(max_depth=6, random_state=42)
final_model.fit(X_train, y_train)

test_preds = final_model.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)

print("Final test accuracy (checked only once):", round(test_acc, 4))

Final test accuracy (checked only once): 0.6993


<div style="background-color:#1A0D2E; border-left:5px solid #B537F2; padding:15px 20px; border-radius:10px; margin-top:10px;">
<h2 style="color:#FFFFFF; font-size:24px; text-shadow:0 0 6px #bb7adc; margin:0;">
Step 4: What Would Go Wrong If I Tuned on the Test Set?
</h2>
</div>

<div style="background-color:#1A0D2E; border-left:7px solid #B537F2; padding:20px; border-radius:14px;">
<p style="color:#FFFFFF; font-size:17px; line-height:1.8;">
If I had checked the test set every time I tried a different
<code>max_depth</code> (instead of only the validation set), I would
have eventually picked the setting that happens to perform best on
that specific test set — not the setting that generalizes best.
At that point, the test accuracy would no longer be an honest estimate
of real-world performance, because I would have indirectly "leaked"
information from the test set into my model choice. The final reported
score would look better than it actually deserves, and the model could
perform worse than expected on truly new, unseen data.
</p>
</div>